In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow as tf
import tensorflow_hub as hub

2026-03-21 11:35:37.127472: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-21 11:35:37.845204: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-21 11:35:41.665142: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

In [3]:
tf.config.list_physical_devices("GPU")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [11]:
batch=50
raw_val_data=tf.keras.utils.text_dataset_from_directory("/mnt/d/Python/DeepLearningGPU/Project_Basic_Text_Classification/stack_overflow_16k/train",batch_size=batch,subset="validation",seed=42,validation_split=0.2)
raw_train_data=tf.keras.utils.text_dataset_from_directory("/mnt/d/Python/DeepLearningGPU/Project_Basic_Text_Classification/stack_overflow_16k/train",batch_size=batch,subset="training",seed=42,validation_split=0.2)
raw_test_data=tf.keras.utils.text_dataset_from_directory("/mnt/d/Python/DeepLearningGPU/Project_Basic_Text_Classification/stack_overflow_16k/test",batch_size=batch)

Found 8000 files belonging to 4 classes.
Using 1600 files for validation.
Found 8000 files belonging to 4 classes.
Using 6400 files for training.
Found 8000 files belonging to 4 classes.


In [12]:
autotune=tf.data.AUTOTUNE
raw_val_data=raw_val_data.cache().prefetch(buffer_size=autotune)
raw_train_data=raw_train_data.cache().prefetch(buffer_size=autotune)
raw_test_data=raw_test_data.cache().prefetch(buffer_size=autotune)

In [13]:
url="https://www.kaggle.com/models/google/universal-sentence-encoder/TensorFlow2/universal-sentence-encoder/2"
hub_layer=hub.KerasLayer(url,input_shape=[],dtype=tf.string,trainable=False)

In [14]:
model=tf.keras.Sequential()
model.add(hub_layer)
model.add(tf.keras.layers.Dense(units=128,activation="relu"))
model.add(tf.keras.layers.Dropout(0.5))
model.add(tf.keras.layers.Dense(units=64,activation="relu"))
model.add(tf.keras.layers.Dropout(0.3))
model.add(tf.keras.layers.Dense(units=16,activation="relu"))
model.add(tf.keras.layers.Dropout(0.1))
model.add(tf.keras.layers.Dense(4,activation="softmax"))
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 keras_layer (KerasLayer)    (None, 512)               256797824 
                                                                 
 dense (Dense)               (None, 128)               65664     
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 64)                8256      
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense_2 (Dense)             (None, 16)                1040      
                                                                 
 dropout_2 (Dropout)         (None, 16)                0

In [15]:
model.compile(optimizer="adam",
              loss=tf.losses.SparseCategoricalCrossentropy(from_logits=False),
              metrics=["accuracy"])

In [16]:
callback=tf.keras.callbacks.EarlyStopping(patience=3,
                                          monitor="val_accuracy",
                                          restore_best_weights=True)

In [17]:
tf.keras.backend.clear_session()
import gc
gc.collect()
model.fit(raw_train_data,validation_data=raw_val_data,epochs=20,callbacks=callback)

Epoch 1/20
128/128 [==============================] - 32s 208ms/step - loss: 1.0586 - accuracy: 0.5669 - val_loss: 0.5070 - val_accuracy: 0.8156
Epoch 2/20
128/128 [==============================] - 13s 105ms/step - loss: 0.5891 - accuracy: 0.7872 - val_loss: 0.4231 - val_accuracy: 0.8313
Epoch 3/20
128/128 [==============================] - 16s 128ms/step - loss: 0.4956 - accuracy: 0.8172 - val_loss: 0.4068 - val_accuracy: 0.8400
Epoch 4/20
128/128 [==============================] - 13s 102ms/step - loss: 0.4691 - accuracy: 0.8245 - val_loss: 0.4038 - val_accuracy: 0.8413
Epoch 5/20
128/128 [==============================] - 13s 105ms/step - loss: 0.4501 - accuracy: 0.8367 - val_loss: 0.4005 - val_accuracy: 0.8469
Epoch 6/20
128/128 [==============================] - 11s 86ms/step - loss: 0.4241 - accuracy: 0.8469 - val_loss: 0.4042 - val_accuracy: 0.8450
Epoch 7/20
128/128 [==============================] - 11s 85ms/step - loss: 0.4002 - accuracy: 0.8566 - val_loss: 0.4109 - val_accu

In [18]:
model.evaluate(raw_test_data)

160/160 [==============================] - 11s 67ms/step - loss: 0.4329 - accuracy: 0.8351


[0.43285539746284485, 0.8351250290870667]